Team coursework: XGBoost classification on an anonymised Kaggle-style tabular dataset (class-imbalanced binary target), part of the same assignment as `svm_imbalance_and_tuning.ipynb` in this folder. The input CSV and rendered outputs are not included, and results are not independently reproduced in this portfolio copy.


In [ ]:
!pip install optuna

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix, average_precision_score
from sklearn.impute import SimpleImputer
from imblearn.ensemble import EasyEnsembleClassifier
import seaborn as sns

from imblearn.ensemble import BalancedBaggingClassifier
from imblearn.over_sampling import SMOTE
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score, recall_score
from sklearn.metrics import accuracy_score, precision_score, f1_score, recall_score, confusion_matrix
from sklearn.model_selection import GridSearchCV

from sklearn.metrics import (precision_recall_curve, auc as auc_fn,
                             f1_score, precision_score, recall_score,
                             matthews_corrcoef, confusion_matrix, accuracy_score)
from sklearn.metrics import precision_recall_fscore_support
from sklearn.feature_selection import SelectFromModel
import optuna
import warnings
warnings.filterwarnings("ignore")

In [ ]:
file_name = "data/datafile_small.csv"
crdata = pd.read_csv(file_name)
crdata.head()

In [ ]:
# drop Var_45 and Id column

crdata = crdata.drop(columns = ['ID_code','var_45'])
crdata.head()

In [ ]:
#Split the data into train, test
random_seed = 30
X = crdata.drop('target', axis=1)
y = crdata['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state = random_seed )


In [ ]:
from sklearn.impute import SimpleImputer

col_with_na = X_train.columns[X_train.isnull().any()].tolist()

if col_with_na:
    imputer = SimpleImputer(strategy='mean')
    X_train[col_with_na] = imputer.fit_transform(X_train[col_with_na])
    X_test[col_with_na] = imputer.transform(X_test[col_with_na])


In [ ]:
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.ensemble import EasyEnsembleClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    fbeta_score, matthews_corrcoef, confusion_matrix,
    precision_recall_curve, auc
)
import pandas as pd

# ---------------------------------------------------------
# BASE XGBOOST PARAMETERS
# ---------------------------------------------------------
BASE_XGB_PARAMS = dict(
    max_depth=6,
    learning_rate=0.1,
    n_estimators=250,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=30,
    eval_metric='logloss',
    n_jobs=-1
)

In [ ]:
# ---------------------------------------------------------
# METRIC FUNCTIONS
# ---------------------------------------------------------
def pr_auc(y_true, y_scores):
    # average_precision_score directly computes the Area Under the Precision-Recall Curve (PRAUC)
    return average_precision_score(y_true, y_scores)

def f2_score(y_true, y_pred):
    return fbeta_score(y_true, y_pred, beta=2)

def evaluate_model(model, X_t, y_t):
    y_pred = model.predict(X_t)

    try:
        y_prob = model.predict_proba(X_t)[:, 1]
    except:
        try:
            y_prob = model.decision_function(X_t)
        except:
            y_prob = y_pred  # fallback for models without probabilities

    # Return all performance metrics
    return {
        'accuracy': accuracy_score(y_t, y_pred),
        'precision': precision_score(y_t, y_pred, zero_division=0),
        'recall': recall_score(y_t, y_pred),
        'f1': f1_score(y_t, y_pred),
        'f2': f2_score(y_t, y_pred),
        'mcc': matthews_corrcoef(y_t, y_pred),
        'prauc': pr_auc(y_t, y_prob),
        'rocauc': roc_auc_score(y_t, y_prob),
        'confusion_matrix': confusion_matrix(y_t, y_pred)
    }

# ---------------------------------------------------------
# 1. BASELINE XGBOOST (No Balancing)
# ---------------------------------------------------------
xgb_base = XGBClassifier(**BASE_XGB_PARAMS)
xgb_base.fit(X_train, y_train)
metrics_base = evaluate_model(xgb_base, X_test, y_test)
print("Baseline metrics:", metrics_base)

# ---------------------------------------------------------
# 2. SMOTE + XGBOOST
# ---------------------------------------------------------
sm = SMOTE(random_state=30)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

xgb_sm = XGBClassifier(**BASE_XGB_PARAMS)
xgb_sm.fit(X_train_sm, y_train_sm)
metrics_sm = evaluate_model(xgb_sm, X_test, y_test)
print("SMOTE metrics:", metrics_sm)

# ---------------------------------------------------------
# 3. CLASS WEIGHT XGBOOST (scale_pos_weight)
# ---------------------------------------------------------
pos_ratio = y_train.value_counts()[0] / y_train.value_counts()[1]

xgb_cw = XGBClassifier(**BASE_XGB_PARAMS, scale_pos_weight=pos_ratio)
xgb_cw.fit(X_train, y_train)
metrics_cw = evaluate_model(xgb_cw, X_test, y_test)
print("Class-weight metrics:", metrics_cw)

# ---------------------------------------------------------
# 4. EASY ENSEMBLE + XGBOOST
# ---------------------------------------------------------
eec = EasyEnsembleClassifier(
    estimator=XGBClassifier(**{**BASE_XGB_PARAMS, 'n_estimators':150}),
    n_estimators=10,
    random_state=30,
    n_jobs=-1
)

eec.fit(X_train, y_train)
metrics_eec = evaluate_model(eec, X_test, y_test)
print("EasyEnsemble metrics:", metrics_eec)

# ---------------------------------------------------------
# SUMMARY TABLE
# ---------------------------------------------------------
rows = []
for name, m in [
    ('Baseline', metrics_base),
    ('SMOTE', metrics_sm),
    ('ClassWeight', metrics_cw),
    ('EasyEnsemble', metrics_eec)
]:
    rows.append({
        'model': name,
        'PRAUC': m['prauc'],
        'ROC-AUC': m['rocauc'],
        'F2': m['f2'],
        'MCC': m['mcc'],
        'F1': m['f1'],
        'Recall': m['recall'],
        'Precision': m['precision'],
        'Accuracy': m['accuracy']
    })

results_df = pd.DataFrame(rows).set_index('model').round(4)
print("\nSummary metrics:\n", results_df)


In [ ]:
# selection rule: highest PRAUC → then highest F2 → then highest MCC
best_by_priority = results_df.sort_values(
    ['PRAUC', 'F2', 'MCC'],
    ascending=False
).index[0]

print("Best model (PRAUC > F2 > MCC):", best_by_priority)

# map name -> estimator object
model_map = {
    'Baseline': xgb_base,
    'SMOTE': xgb_sm,
    'ClassWeight': xgb_cw,
    'EasyEnsemble': eec
}

best_model = model_map[best_by_priority]


In [ ]:
from sklearn.feature_selection import SelectFromModel

# Use XGBoost to compute feature importance (only XGB has attribute .feature_importances_)
# If the best model is EasyEnsemble or SMOTE etc., we still use XGBoost for importance.
temp = XGBClassifier(**BASE_XGB_PARAMS)
temp.fit(X_train, y_train)

# extract importances
importances = pd.Series(temp.feature_importances_, index=X_train.columns)

# number of features to keep
top_k = 50
top_features = importances.sort_values(ascending=False).head(top_k).index.tolist()

# filter train/test with the selected features
X_train_fs = X_train[top_features].copy()
X_test_fs = X_test[top_features].copy()

print(f"Selected top {len(top_features)} features based on XGBoost importances.")


In [ ]:
# ---------------------------------------------------------
# Train the BEST model on selected features
# ---------------------------------------------------------
# X_train_fs and X_test_fs already have top_k features

# Retrieve the best model object
best_model_name = best_by_priority
best_model_obj = model_map[best_model_name]

# Clone the model to avoid overwriting original
from sklearn.base import clone
best_model_fs = clone(best_model_obj)

# Fit the model on the reduced feature set
best_model_fs.fit(X_train_fs, y_train)

# Evaluate metrics
metrics_best_fs = evaluate_model(best_model_fs, X_test_fs, y_test)
print(f"\nMetrics of BEST model ({best_model_name}) after feature selection:")
print(metrics_best_fs)

# ---------------------------------------------------------
# Compare metrics: before vs after feature selection
# ---------------------------------------------------------
print(f"\nComparison of {best_model_name}:")
for metric in ['accuracy','precision','recall','f1','f2','mcc','prauc']:
    before = model_map[best_model_name]  # original trained model
    before_val = evaluate_model(before, X_test, y_test)[metric]
    after_val = metrics_best_fs[metric]
    print(f"{metric}: Before FS = {before_val:.4f} | After FS = {after_val:.4f}")


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, precision_recall_curve, auc
from sklearn.base import clone

# -----------------------------
# Define PRAUC scorer
# -----------------------------
def pr_auc_scorer(y_true, y_scores):
    return average_precision_score(y_true, y_scores)

prauc_scorer = make_scorer(pr_auc_scorer, needs_proba=True)

# -----------------------------
# Clone the best model (EasyEnsemble)
# -----------------------------
ee_model = clone(eec)  # original trained on full feature set

# -----------------------------
# Define parameter grid
# -----------------------------
param_grid = {
    'n_estimators': [5, 10, 15],  # number of ensembles
    'estimator__n_estimators': [100, 150],  # number of trees in base XGB
    'estimator__max_depth': [4, 6],
    'estimator__learning_rate': [0.05, 0.1]
}

# -----------------------------
# GridSearchCV
# -----------------------------
grid = GridSearchCV(
    estimator=ee_model,
    param_grid=param_grid,
    scoring=prauc_scorer,
    cv=3,
    n_jobs=-1,
    verbose=2
)

# -----------------------------
# Fit GridSearchCV on full features
# -----------------------------
grid.fit(X_train, y_train)

# -----------------------------
# Best parameters & score
# -----------------------------
print("Best parameters:", grid.best_params_)
print("Best PRAUC:", grid.best_score_)

# -----------------------------
# Evaluate best estimator on test set
# -----------------------------
best_ee = grid.best_estimator_
metrics_best_ee = evaluate_model(best_ee, X_test, y_test)
print("\nMetrics of best EasyEnsemble after GridSearchCV:")
print(metrics_best_ee)


In [ ]:
# -----------------------------------------------------------
# 📌 OPTUNA FOR EASYENSEMBLE + XGBOOST (BEFORE FEATURE SELECTION)
#     Optimizing PRAUC using Stratified 3-Fold CV
# -----------------------------------------------------------

import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from imblearn.ensemble import EasyEnsembleClassifier
from xgboost import XGBClassifier
import numpy as np

# 🎯 Objective function for Optuna
def objective(trial):

    # 🔧 Suggest hyperparameters
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 400),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 5.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 5.0),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 15.0),
        'objective': 'binary:logistic',
        'tree_method': 'hist'  # speed boost
    }

    # 🧠 Use XGB inside EasyEnsemble
    model = EasyEnsembleClassifier(
        estimator=XGBClassifier(**params),
        n_estimators=10  # Default; can be tuned later
    )

    # 📌 Stratified 3-fold CV
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=30)

    pr_scores = []

    # 📌 Perform CV manually to compute PRAUC
    for train_idx, valid_idx in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]

        model.fit(X_tr, y_tr)
        preds = model.predict_proba(X_val)[:, 1]
        pr = average_precision_score(y_val, preds)
        pr_scores.append(pr)

    return np.mean(pr_scores)  #maximize PRAUC


# 🚀 Run Optuna Study
study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=30))
study.optimize(objective, n_trials=50, show_progress_bar=True)

# 📌 Best Hyperparameters
print("\n💡 Best PRAUC:", study.best_value)
print("🏆 Best Params:", study.best_params)


In [ ]:
best_params = {
    'n_estimators': 371,
    'learning_rate': 0.06492342931525268,
    'max_depth': 10,
    'subsample': 0.5011594943518456,
    'colsample_bytree': 0.9979288577841255,
    'reg_alpha': 1.572466757521283,
    'reg_lambda': 1.1592167250357228,
    'scale_pos_weight': 1.136181321463428}


In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_recall_curve, auc, fbeta_score, matthews_corrcoef,
    accuracy_score, precision_score, recall_score, f1_score,
    average_precision_score
) # Added average_precision_score for consistency with original notebook pr_auc
import joblib
import numpy as np

# Train Final Model as per best Optuna
final_model = XGBClassifier(
    n_estimators=371,
    learning_rate=0.06492342931525268,
    max_depth=10,
    subsample=0.5011594943518456,
    colsample_bytree=0.9979288577841255,
    reg_alpha=1.572466757521283,
    reg_lambda=1.1592167250357228,
    scale_pos_weight=1.136181321463428,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=30,
    use_label_encoder=False
)

final_model.fit(X_train, y_train)

# Predictions
y_pred = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)[:, 1]

# ==== METRICS ====
# PRAUC
prauc = average_precision_score(y_test, y_proba)

# F2 Score
f2 = fbeta_score(y_test, y_pred, beta=2)

# MCC
mcc = matthews_corrcoef(y_test, y_pred)

# Print Results
print("\n---- FINAL MODEL PERFORMANCE ----")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")
print(f"F2 Score:  {f2:.4f}")
print(f"MCC:       {mcc:.4f}")
print(f"PRAUC:     {prauc:.4f}")
print(f"ROC AUC:   {roc_auc_score(y_test, y_proba):.4f}")

print("\n---- Confusion Matrix ----")
print(confusion_matrix(y_test, y_pred))

# Save Model
joblib.dump(final_model, "final_xgboost_model.pkl")
print("\nModel saved as 'final_xgboost_model.pkl'")

In [ ]:
from xgboost import XGBClassifier
from imblearn.ensemble import EasyEnsembleClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, fbeta_score,
    matthews_corrcoef, roc_auc_score, precision_recall_curve, auc, confusion_matrix, average_precision_score
)
import joblib

# ===== CREATE BASE XGBOOST ESTIMATOR =====
base_estimator = XGBClassifier(
    n_estimators=371,
    learning_rate=0.06492342931525268,
    max_depth=10,
    subsample=0.5011594943518456,
    colsample_bytree=0.9979288577841255,
    reg_alpha=1.572466757521283,
    reg_lambda=1.1592167250357228,
    scale_pos_weight=1.136181321463428,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=random_seed,
    use_label_encoder=False
)


# ===== FINAL EASYENSEMBLE MODEL =====
final_model = EasyEnsembleClassifier(
    estimator=base_estimator,
    n_estimators=15,
    random_state=random_seed,
    n_jobs=-1
)

# Train
final_model.fit(X_train, y_train)

# Predictions
y_pred = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)[:, 1]

# ===== METRICS =====
prauc = average_precision_score(y_test, y_proba)
f2 = fbeta_score(y_test, y_pred, beta=2)
mcc = matthews_corrcoef(y_test, y_pred)

print("\n---- FINAL MODEL PERFORMANCE ----")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")
print(f"F2 Score:  {f2:.4f}")
print(f"MCC:       {mcc:.4f}")
print(f"PRAUC:     {prauc:.4f}")
print(f"ROC AUC:   {roc_auc_score(y_test, y_proba):.4f}")

print("\n---- Confusion Matrix ----")
print(confusion_matrix(y_test, y_pred))

# Save Model
joblib.dump(final_model, "final_easyensemble_model.pkl")
print("\nModel saved as 'final_easyensemble_model.pkl'")
